# PR Review Env: GRPO Training (Kaggle)

This notebook clones your GitHub repo, installs training dependencies, starts the OpenEnv server, runs GRPO training, and saves artifacts for submission evidence.

## 1) Runtime checks
Make sure Kaggle Notebook accelerator is set to **GPU**.

In [ ]:
import os
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA capability:', torch.cuda.get_device_capability(0))
else:
    print('No GPU detected. Enable GPU in Kaggle settings.')

## 2) Clone your GitHub project
Replace `REPO_URL` with your uploaded repo URL.

In [ ]:
REPO_URL = "https://github.com/mithilesh11705/Meta_3_1.git"
!git clone "$REPO_URL" repo
%cd repo

## 3) Optional: Hugging Face token from Kaggle Secrets
Create a Kaggle secret named `HF_TOKEN` if your model needs auth.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = token
    print("HF_TOKEN loaded from Kaggle Secrets")
except Exception:
    print("HF_TOKEN not set (ok for public models)")

## 4) Install dependencies

In [ ]:
!pip install -q -r requirements-train.txt

## 5) Start OpenEnv server inside notebook

In [ ]:
import subprocess
import time

# Important: avoid stdout=PIPE here, otherwise uvicorn logs can fill the pipe and block requests.
server = subprocess.Popen(
    ["python", "-m", "uvicorn", "server.app:app", "--host", "0.0.0.0", "--port", "7860"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)
time.sleep(6)
print("Server PID:", server.pid)
!curl -s http://127.0.0.1:7860/health

## 6) Run GRPO training (Kaggle-friendly starter config)
This uses a smaller model/config first so the run finishes reliably on Kaggle GPU.

In [ ]:
!python train_grpo.py \
  --env-base-url http://127.0.0.1:7860 \
  --model-name Qwen/Qwen2.5-0.5B-Instruct \
  --num-samples 96 \
  --num-train-epochs 2 \
  --per-device-train-batch-size 1 \
  --gradient-accumulation-steps 4 \
  --num-generations 4 \
  --episodes-per-task 1 \
  --max-episode-steps 3 \
  --eval-tasks-per-difficulty 1 \
  --skip-initial-eval \
  --skip-post-eval \
  --env-timeout-seconds 120 \
  --env-max-retries 3 \
  --max-completion-length 128 \
  --max-new-tokens 128 \
  --output-dir artifacts/grpo_kaggle_run

## 7) View training outputs

In [ ]:
from IPython.display import Image, Markdown, display
import json

display(Image(filename='artifacts/grpo_kaggle_run/plots/reward_curve.png'))
display(Markdown(open('artifacts/grpo_kaggle_run/logs/before_after.md', 'r', encoding='utf-8').read()))
summary = json.load(open('artifacts/grpo_kaggle_run/logs/training_summary.json', 'r', encoding='utf-8'))
summary

## 8) Export artifacts for download/submission

In [ ]:
!zip -r grpo_kaggle_artifacts.zip artifacts/grpo_kaggle_run
!ls -lh grpo_kaggle_artifacts.zip